## Data Preparation and Exploratory Data Analysis

## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Load Dataset

In [ ]:
DATA_PATH = "../data/Dataset_Eating_Disorder.csv"

df_raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("Shape:", df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

## Dataset Structure

In [ ]:
demographic_features = [
    "Age_Range", "Gender", "Education_Level", "Employment_Status", "Marital_Status"
]

self_perception_feature = "Perception_EatingDisorder"

print("Observations:", df_raw.shape[0])
print("Variables:", df_raw.shape[1])
print("Demographic variables:", len(demographic_features))
print("Behavioral variables:", df_raw.shape[1] - len(demographic_features) - 1)
print("Self-perception variable:", self_perception_feature)

## Data Quality

In [ ]:
print("Missing values:", df_raw.isna().sum().sum())
print("Duplicate rows:", df_raw.duplicated().sum())

columns_with_spaces = [column for column in df_raw.columns if column != column.strip() or " _" in column or "_ " in column]
print("Columns with spacing problems:", columns_with_spaces)

In [ ]:
column_rename_map = {
    "DesireToBuy _FromSnackBarOrCafe": "DesireToBuy_FromSnackBarOrCafe"
}

df = df_raw.rename(columns=column_rename_map).copy()

print(column_rename_map)
print("Corrected column exists:", "DesireToBuy_FromSnackBarOrCafe" in df.columns)

## Duplicate Analysis

In [ ]:
duplicate_mask = df.duplicated(keep="first")
df_unique = df.loc[~duplicate_mask].copy()

print("Original dataset shape:", df.shape)
print("Duplicate rows:", duplicate_mask.sum())
print("Unique dataset shape:", df_unique.shape)

## Behavioral Features

In [ ]:
behavioral_groups = {
    "Restrained / Weight-control Eating": [
        "EatLess_OnWeightGain", "EatLess_AtMealtime", "RefuseFood_WeightConcern", "Monitor_Food",
        "Eat_SlimmingFoods", "EatLess_AfterOvereating", "EatLess_ToPreventWeightGain",
        "AvoidSnacks_BetweenMealsToWatchWeight", "AvoidEveningEating_ToWatchWeight",
        "ConsiderWeight_WhenEating"
    ],
    "Emotional Eating": [
        "Eat_WhenIrritated", "Eat_WhenUnoccupied", "Eat_WhenDepressedOrDiscouraged", "Eat_WhenLonely",
        "Eat_WhenSomeoneLetDown", "Eat_WhenAngry", "Eat_WhenExpectingBad", "Eat_WhenAnxious",
        "Eat_WhenThingsGoWrong", "Eat_WhenFrightened", "Eat_WhenDisappointed",
        "Eat_WhenEmotionallyUpset", "Eat_WhenBoredOrRestless"
    ],
    "External / Food-cue Eating": [
        "EatMore_IfFoodTasty", "EatMore_IfFoodSmellsOrLooksGood", "Eat_WhenSeeDeliciousFood",
        "Eat_DeliciousFoodImmediately", "DesireToBuy_FromBakery", "DesireToBuy_FromSnackBarOrCafe",
        "DesireToEat_WhenSeeOthersEating", "Resist_DeliciousFood", "EatMore_WhenSeeOthersEating",
        "Eat_WhenPreparingMeal"
    ],
    "Body Image / Weight-Shape Concern": [
        "Days_DesireForFlatStomach", "Days_FeltFat", "Days_WeightAffectedSelfJudgment",
        "Days_ShapeAffectedSelfJudgment", "Days_DissatisfiedWithWeight", "Days_DissatisfiedWithShape",
        "Days_UncomfortableToSeeOwnBody", "Days_UncomfortableBecauseOthersSeeingShape",
        "Days_TriedLimitFoodControlShapeOrWeight", "Days_FastedControlShapeOrWeight",
        "Days_ExcludedFoodControlShapeOrWeight", "Days_FollowedRulesControlShapeOrWeight",
        "Days_FearLosingControlOverEating"
    ],
    "Habitual Eating": [
        "Eat_SpecificFoodsHabitually", "Location_TriggersHabitualEating",
        "AutomaticEating_WhenExperiencingStrongEmotion", "Realize_AfterEatingOutOfHabit"
    ]
}

behavioral_features = [question for questions in behavioral_groups.values() for question in questions]

group_summary = pd.Series({group: len(questions) for group, questions in behavioral_groups.items()}, name="Questions")
print(group_summary)
print("Total behavioral questions:", len(behavioral_features))

## Ordinal Encoding

In [ ]:
scale_a_features = [question for question in behavioral_features if not question.startswith("Days_")]
scale_b_features = [question for question in behavioral_features if question.startswith("Days_")]

scale_a_mapping = {
    "Never": 0, "Seldom": 1, "Sometimes": 2, "Often": 3, "Very often": 4
}

scale_b_mapping = {
    "No days": 0, "1-5 days": 1, "6-12 days": 2, "13-15 days": 3, "Every day": 4
}

print("Scale A questions:", len(scale_a_features))
print("Scale B questions:", len(scale_b_features))
print(scale_a_mapping)
print(scale_b_mapping)

In [ ]:
df_processed = df_unique.copy()

for column in scale_a_features:
    df_processed[column] = df_processed[column].map(scale_a_mapping)

for column in scale_b_features:
    df_processed[column] = df_processed[column].map(scale_b_mapping)

behavioral_data = df_processed[behavioral_features].copy()

behavioral_data.head()

In [ ]:
group_means = pd.Series({
    group: behavioral_data[questions].mean().mean()
    for group, questions in behavioral_groups.items()
})

group_means.sort_values().plot(kind="barh", color="darkorange")
plt.title("Mean Encoded Response by Behavioral Group")
plt.xlabel("Mean encoded response (0-4)")
plt.ylabel("")
plt.xlim(0, 4)
plt.tight_layout()
plt.show()

## Conclusion

- The raw dataset contains 700 observations and 56 variables.
- The main data-quality issues are 99 exact duplicate rows and one column-name spacing error.
- The 601 unique full-record patterns are retained as the primary data for later analysis.
- All 50 behavioral questions are preserved and encoded with the original two ordinal mappings.
- Demographic variables are excluded from the behavioral matrix.
- No clustering, question reduction, model selection, or intermediate artifact is produced here.